In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/cleaned/supply_chain_cleaned.csv')
region_risk = pd.read_csv('../data/cleaned/delivery_risk_by_region.csv')
shipping_risk = pd.read_csv('../data/cleaned/shipping_mode_risk.csv')
product_risk = pd.read_csv('../data/cleaned/product_profitability_risk.csv')

print("Main data:", df.shape)
print("Region risk:", region_risk.shape)
print("Shipping risk:", shipping_risk.shape)
print("Product risk:", product_risk.shape)

Main data: (180519, 29)
Region risk: (23, 20)
Shipping risk: (4, 17)
Product risk: (118, 10)


In [3]:
# ─────────────────────────────────────────────
# KPI SUMMARY TABLE
# ─────────────────────────────────────────────

kpi = {
    'total_orders': df['order_id'].nunique(),
    'total_revenue': round(df['sales'].sum(), 2),
    'total_profit': round(df['order_profit_per_order'].sum(), 2),
    'profit_margin_pct': round(df['order_profit_per_order'].sum() / df['sales'].sum() * 100, 2),
    'avg_order_value': round(df['sales'].mean(), 2),
    'avg_shipping_delay_days': round(df['shipping_delay_days'].mean(), 2),
    'late_delivery_rate_pct': round(df['late_delivery_risk'].mean() * 100, 2),
    'on_time_delivery_rate_pct': round((1 - df['late_delivery_risk'].mean()) * 100, 2),
    'total_revenue_at_risk': round(df[df['late_delivery_risk'] == 1]['sales'].sum(), 2),
    'weighted_revenue_at_risk': round(df['weighted_revenue_at_risk'].sum(), 2),
    'loss_orders': int(df['is_loss_order'].sum()),
    'loss_order_rate_pct': round(df['is_loss_order'].mean() * 100, 2)
}

kpi_df = pd.DataFrame([kpi])
kpi_df.to_csv('../data/cleaned/kpi_summary.csv', index=False)

print("KPI Summary:")
for key, value in kpi.items():
    print(f"  {key}: {value}")

KPI Summary:
  total_orders: 65752
  total_revenue: 36784735.01
  total_profit: 3966902.97
  profit_margin_pct: 10.78
  avg_order_value: 203.77
  avg_shipping_delay_days: 0.57
  late_delivery_rate_pct: 54.83
  on_time_delivery_rate_pct: 45.17
  total_revenue_at_risk: 20126395.27
  weighted_revenue_at_risk: 22873934.25
  loss_orders: 33784
  loss_order_rate_pct: 18.71


In [4]:
# ─────────────────────────────────────────────
# ACTION PRIORITY TABLE
# ─────────────────────────────────────────────

actions = []

# High risk regions
high_risk_regions = region_risk[region_risk['risk_flag'] == 'HIGH RISK']
for _, row in high_risk_regions.iterrows():
    priority_score = round(
        row['revenue_at_risk'] * row['risk_score'] * (1 + row['avg_delay_days'] / 10), 2
    )
    trend_note = f" Risk is {row.get('risk_trend', 'Stable →')}."
    actions.append({
        'priority': 'HIGH',
        'area': 'Regional Delivery',
        'issue': (
            f"{row['order_region']} ({row['market']}) has {row['late_rate_pct']}% late delivery rate "
            f"with ${round(row['revenue_at_risk'], 0):,.0f} revenue at risk and risk score {row['risk_score']}.{trend_note}"
        ),
        'revenue_at_risk': round(row['revenue_at_risk'], 2),
        'risk_score': row['risk_score'],
        'risk_trend': row.get('risk_trend', 'Stable →'),
        'priority_score': priority_score,
        'action': (
            f"Flag high-value orders in {row['order_region']} for priority routing review. "
            f"Investigate root cause with logistics team. "
            f"If late rate remains above 40% over the next review cycle, consider evaluating alternative logistics partners for this lane."
        )
    })

# High risk shipping modes
high_risk_shipping = shipping_risk[shipping_risk['risk_flag'] == 'HIGH RISK']
for _, row in high_risk_shipping.iterrows():
    priority_score = round(
        row['revenue_at_risk'] * row['risk_score'] * (1 + row['avg_delay_days'] / 10), 2
    )
    actions.append({
        'priority': 'HIGH',
        'area': 'Shipping Mode',
        'issue': (
            f"{row['shipping_mode']} has {row['late_rate_pct']}% late rate "
            f"with avg delay of {round(row['avg_delay_days'], 1)} days "
            f"and ${round(row['revenue_at_risk'], 0):,.0f} revenue at risk."
        ),
        'revenue_at_risk': round(row['revenue_at_risk'], 2),
        'risk_score': row['risk_score'],
        'risk_trend': 'N/A',
        'priority_score': priority_score,
        'action': (
            f"Review order routing policy for {row['shipping_mode']}. "
            f"Consider moving high-value orders to a faster shipping tier. "
            f"Raise findings with logistics team for SLA review."
        )
    })

# Loss making products
loss_products = product_risk[product_risk['risk_tag'] == 'Loss Making'].head(10)
for _, row in loss_products.iterrows():
    priority_score = round(abs(row['total_profit']) * 10, 2)
    actions.append({
        'priority': 'MEDIUM',
        'area': 'Product Profitability',
        'issue': (
            f"{row['product_name']} ({row['category_name']}) is loss making. "
            f"Total profit: ${round(row['total_profit'], 0):,.0f}. "
            f"Margin: {row['profit_margin_pct']}%. Loss rate: {row['loss_rate_pct']}% of orders."
        ),
        'revenue_at_risk': round(abs(row['total_profit']), 2),
        'risk_score': 0,
        'risk_trend': 'N/A',
        'priority_score': priority_score,
        'action': (
            f"Review discount policy on {row['product_name']}. "
            f"If margin cannot reach a positive level after discount review, consider flagging for discontinuation. "
            f"Avoid scaling order volume on this product until profitability is addressed."
        )
    })

# Volume trap products
volume_traps = product_risk[product_risk['risk_tag'] == 'Volume Trap'].head(10)
for _, row in volume_traps.iterrows():
    priority_score = round(row['total_quantity'] * (5 - row['profit_margin_pct']), 2)
    actions.append({
        'priority': 'MEDIUM',
        'area': 'Volume Trap',
        'issue': (
            f"{row['product_name']} has {int(row['total_quantity']):,} units sold "
            f"but only {row['profit_margin_pct']}% margin. "
            f"High volume is masking low profitability."
        ),
        'revenue_at_risk': round(row['total_revenue'] * 0.05, 2),
        'risk_score': 0,
        'risk_trend': 'N/A',
        'priority_score': priority_score,
        'action': (
            f"Review discount structure on {row['product_name']}. "
            f"At this order volume, even a small margin improvement has meaningful impact. "
            f"Consider testing a modest price adjustment or bundling with a higher-margin product."
        )
    })

action_df = pd.DataFrame(actions)
action_df = action_df.sort_values('priority_score', ascending=False).reset_index(drop=True)
action_df['rank'] = action_df.index + 1

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
action_df.to_csv('../data/cleaned/action_priority_table.csv', index=False)
print(f"action_priority_table.csv saved. Total actions: {len(action_df)}")
print("\nTop 5 priority actions:")
print(action_df[['rank', 'priority', 'area', 'priority_score']].head(5))



print("Top actions:")
print(action_df[['rank', 'priority', 'area', 'priority_score', 'revenue_at_risk']].to_string())

action_priority_table.csv saved. Total actions: 4

Top 5 priority actions:
   rank priority                   area  priority_score
0     1     HIGH      Regional Delivery  269,046,966.65
1     2   MEDIUM  Product Profitability        9,651.20
2     3   MEDIUM  Product Profitability        2,559.50
3     4   MEDIUM  Product Profitability        1,695.60
Top actions:
   rank priority                   area  priority_score  revenue_at_risk
0     1     HIGH      Regional Delivery  269,046,966.65     3,292,013.64
1     2   MEDIUM  Product Profitability        9,651.20           965.12
2     3   MEDIUM  Product Profitability        2,559.50           255.95
3     4   MEDIUM  Product Profitability        1,695.60           169.56


In [5]:
print("Checking all output files:")

import os
files = [
    '../data/cleaned/supply_chain_cleaned.csv',
    '../data/cleaned/delivery_risk_by_region.csv',
    '../data/cleaned/shipping_mode_risk.csv',
    '../data/cleaned/product_profitability_risk.csv',
    '../data/cleaned/kpi_summary.csv',
    '../data/cleaned/action_priority_table.csv',
]

for f in files:
    exists = os.path.exists(f)
    size = round(os.path.getsize(f) / 1024, 1) if exists else 0
    print(f"  {'✓' if exists else '✗'} {f.split('/')[-1]} — {size} KB")

Checking all output files:
  ✓ supply_chain_cleaned.csv — 50239.8 KB
  ✓ delivery_risk_by_region.csv — 4.5 KB
  ✓ shipping_mode_risk.csv — 0.9 KB
  ✓ product_profitability_risk.csv — 12.7 KB
  ✓ kpi_summary.csv — 0.3 KB
  ✓ action_priority_table.csv — 1.7 KB


In [6]:
action_df = pd.read_csv('../data/cleaned/action_priority_table.csv')

# Short issue summary
def short_issue(row):
    if row['area'] == 'Regional Delivery':
        return f"{row['issue'].split('(')[0].strip()} — {row['issue'].split('late delivery rate')[0].split('has')[1].strip()} late rate"
    elif row['area'] == 'Shipping Mode':
        return f"{row['issue'].split('has')[0].strip()} — {row['issue'].split('has')[1].split('%')[0].strip()}% late rate"
    else:
        product = row['issue'].split('(')[0].strip()
        margin = row['issue'].split('Margin:')[1].split('%')[0].strip() if 'Margin:' in row['issue'] else ''
        return f"{product} — {margin}% margin"

# Short action summary
def short_action(row):
    return row['action'].split('.')[0].strip()

action_df['short_issue'] = action_df.apply(short_issue, axis=1)
action_df['short_action'] = action_df.apply(short_action, axis=1)

action_df.to_csv('../data/cleaned/action_priority_table.csv', index=False)

print("Done. New columns:")
print(action_df[['rank', 'area', 'short_issue', 'short_action']])

Done. New columns:
   rank                   area  \
0     1      Regional Delivery   
1     2  Product Profitability   
2     3  Product Profitability   
3     4  Product Profitability   

                                         short_issue  \
0                  Western Europe — 55.85% late rate   
1                SOLE E35 Elliptical — -3.22% margin   
2  Bushnell Pro X7 Jolt Slope Rangefinder — -3.88...   
3                 SOLE E25 Elliptical — -1.7% margin   

                                        short_action  
0  Flag high-value orders in Western Europe for p...  
1      Review discount policy on SOLE E35 Elliptical  
2  Review discount policy on Bushnell Pro X7 Jolt...  
3      Review discount policy on SOLE E25 Elliptical  


In [7]:
action_df = pd.read_csv('../data/cleaned/action_priority_table.csv')

# Add action type column
def action_type(row):
    if row['priority'] == 'HIGH':
        return 'Escalate'
    elif row['area'] == 'Product Profitability':
        return 'Review'
    elif row['area'] == 'Volume Trap':
        return 'Monitor'
    else:
        return 'Review'

action_df['action_type'] = action_df.apply(action_type, axis=1)

action_df.to_csv('../data/cleaned/action_priority_table.csv', index=False)

print("Done:")
print(action_df[['rank', 'priority', 'area', 'action_type']])

Done:
   rank priority                   area action_type
0     1     HIGH      Regional Delivery    Escalate
1     2   MEDIUM  Product Profitability      Review
2     3   MEDIUM  Product Profitability      Review
3     4   MEDIUM  Product Profitability      Review
